<a href="https://colab.research.google.com/github/ja2faded/ds2002-fa26/blob/main/Copy_of_2026_09_18_%E2%80%94_Pandas_Challenge_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
#revenue = qty multiplied by price
df['revenue'] = df['qty'] * df['price']

#total revenue and units
revenue_total = df['revenue'].sum()
units_total = df['qty'].sum()

print('total revenue ($):', revenue_total)
print('total units sold: ', units_total)

total revenue ($): 8520.0
total units sold:  783


The 400 orders (n rows from the data) generated $ 8520.0 and sold 783 units

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
#group orders by category adn get sum of the revenue in the categories
by_category = df.groupby('category', as_index=False).agg(revenue = ('revenue', 'sum'))

#share percent of each category
by_category['share_percent'] = (by_category['revenue'] / revenue_total * 100)

#sort
by_category = (by_category.sort_values('revenue', ascending=False).reset_index(drop=True))
by_category




,category,revenue,share_percent
0,Food,4293.0,50.387324
1,Merch,1771.5,20.792254
2,Drink,1554.0,18.239437
3,RainGear,901.5,10.580986


By category, Food had the highest revenue with 4293 dollars, followed by Merch with 1771.5 dollars, then Drink with 1554 dollars, and lastly Rain Gear with 901.5 dollars. The share of total as a percentage follows this same descending order (50.4%, 20.8%, 18.2%, 10.6%)

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
#group orders by vendor id
by_vendor = (
    df.groupby('vendor_id', as_index=False)
    .agg(average_order_revenue = ('revenue', 'mean'), order_count= ('revenue','size'))
    .sort_values('average_order_revenue', ascending=False)
    .reset_index(drop=True)
    )
#round to 2 decimals
print(by_vendor.round(2))

#locate top vendor
top_vendor= by_vendor.iloc[0]

print('Top Vendor: ', top_vendor['vendor_id'])


  vendor_id  average_order_revenue  order_count
0      V-01                  22.60           94
1      V-18                  21.75          108
2      V-05                  20.58           93
3      V-10                  20.31          105
Top Vendor:  V-01


V-01 had the highest average order revenue with an overage of $ 22.60 per order over 94 orders. All vendors had roughly the same number of orders (ranging from 93 orders-108 orders).

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
#find merch revenue
merch_revenue = df.loc[df['category'].eq('Merch'), 'revenue'].sum()

#merch share percent to 1 decimal
merch_share = round(merch_revenue / revenue_total * 100, 1)

print(f"Merch revenue share percent: {merch_share}%")


Merch revenue share percent: 20.8%


Merch generated 20.8% of the total revenue.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
#look up table matches vendor ids to names
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

#left join keeps all orders even if no matching vendor ID
joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

#vendor IDs with no match
unmatched_ids = (joined.loc[joined['vendor_name'].isna(),'vendor_id'].unique())

# # of unmatched orders
unmatched_orders = joined['vendor_name'].isna().sum()

#look at revenue and rows before
rows_before = len(df)
revenue_before = df['revenue'].sum()

print('Rows before: ', rows_before)
print('Rows after: ', len(joined))
print('Revenue before: ', revenue_before)
print('Revenue after: ', joined['revenue'].sum())
print('Unmatched ids', unmatched_ids)
print("orders from unmatched ventor: ", unmatched_orders)

#keep unmatched orders under the label name unavailable
joined['vendor_name'] = joined['vendor_name'].fillna(joined['vendor_id'] + ' name unavailable')

Rows before:  400
Rows after:  400
Revenue before:  8520.0
Revenue after:  8520.0
Unmatched ids ['V-18']
orders from unmatched ventor:  108


The unmatched vendor ID was V-18 and they had 108 orders. I used a left join in order to kept the orders as 'name unavailable' instead of discarding the rows since the orders/ revenue from the orders are still valid (I did not want the dats set to lose these values).

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
#pivot table with total rows and columns
vendor_category_pivot = joined.pivot_table(
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

vendor_category_pivot

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
V-18 name unavailable,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


The bottom row (Total) adds up each individual category for all of the vendors. The far right row adds up all categories for each vendor.

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

To the vendors, I would suggest that they ensure they have adequete food, since food generated 4293 dollars of revenue across the vendors and had a 50.39 percent share of the total revenue. Since it generates a largee portion of their revenue, not having enough could hurt their revenue. Additionally, I would say they should ensure there is not a surpluss of stock of Rain Gear under normal weather conditions, since it only generated $901.5 and 10.58% of the revenue. However, its demand may rise with rainy weather, so vendors should adjust their stock accordingly to make sure they do not over supply/ undersupply throughout the season.

Of my 7 answers, I think that question 2's answer is the most misleading. Although it is accurate that it 10.58 percent of the total revenue, it likely fluctuates much more between games than the food or merch since it is very dependent on the weather. Seeing that it only takes up a small percent of the revenue could deter from stocking up, which in the end, could result in the loss of stock at a rainy game.